# Reasoning SLM - Stage 2: SFT

## Introduction

Stage 2 fine-tunes **Qwen2.5-0.5B** (base) with LoRA SFT on the GSM8K train solutions, then scores it zero-shot with the Stage 1 harness:

- Training data: the 6,974 train problems, each as the zero-shot `Question: … Answer:` prompt followed by its reference solution, ending with a final line `#### <number>`.
- Loss on the solution only, with EOS at the end of each completion, so the model learns the format and when to stop.
- The same eval as Stage 1: the 499-problem validation subset, greedy decoding, at most 512 new tokens, the same stop rules, and the 20-prompt general check.

**Prediction from Stage 1:** SFT should bring zero-shot accuracy to about 48% by fixing format and stopping. A clearly higher score would mean SFT adds more than format.

## Setup

Clone the repository so the shared code in `src/` can be imported, and install the requirements. Results are written to `results/stage2/`. Checkpoints, the adapter and the merged model are saved to Google Drive (`/content/drive/MyDrive/slm-post-training`), so mount Drive before running. Download the results after each run, because `/content` is wiped when the runtime disconnects.

In [2]:
import os

REPO_URL = "https://github.com/JuaniLlaberia/slm-post-training.git"
REPO_DIR = "/content/slm-post-training"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

/content/slm-post-training


### Imports

In [ ]:
from src.config import BASE_MODEL, RESULTS_DIR, SEED
from src.data.gsm8k import load_gsm8k
from src.evaluation.general_check import GENERAL
from src.evaluation.generation import generate
from src.evaluation.harness import evaluate, save_results
from src.prompts.few_shot import STOP, build_prompt

import pandas as pd
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

STAGE_DIR = RESULTS_DIR / "stage2"

## Data

The same split as Stage 1: a seeded 499-problem validation subset held out from train, and the full 1,319-problem test set kept for final numbers.

Each example becomes a prompt-completion pair. The prompt is the zero-shot `Question: … Answer:` prompt from Stage 1, and the completion is the cleaned GSM8K solution, ending with `#### <number>`. The leading space matches how the base model continues after `Answer:`. The validation pairs are used for the eval loss during training.

In [4]:
train_dataset, val_dataset, test_dataset = load_gsm8k()

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train dataset size: 6974
Validation dataset size: 499
Test dataset size: 1319


In [10]:
def to_sft(ex):
    """
    Turns a GSM8K example into a prompt-completion pair: the zero-shot prompt and the reference solution.
    """
    return {"prompt": build_prompt(ex["question"]),
            "completion": " " + ex["answer"]}

train_dataset = train_dataset.map(to_sft, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(to_sft, remove_columns=val_dataset.column_names)

Map:   0%|          | 0/6974 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

## Model

LoRA on every linear layer of `Qwen2.5-0.5B`: rank 16, alpha 32, dropout 0.1. That trains 8.8M parameters, 1.75% of the model.

In [15]:
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [18]:
RANK = 16
ALPHA = 32

config = LoraConfig(
    r=RANK,
    lora_alpha=ALPHA,
    target_modules="all-linear",
    lora_dropout=0.1,
    bias="none",
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [24]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## Training

`SFTTrainer` with `completion_only_loss=True`: the loss is computed on the solution only, not on the prompt, and TRL appends the EOS token to each completion. One epoch over the 6,974 train problems is 436 steps at batch size 16, with learning rate 2e-4 (cosine schedule, 25 warmup steps), bf16 and at most 512 tokens per example. The eval loss is computed and a checkpoint saved every 100 steps. At the end, the checkpoint with the lowest eval loss is loaded.

The run below used 2 epochs (872 steps). Epoch 2 only memorized the training set (see the curves below), so the config is set to 1 epoch for anyone rerunning it.

In [26]:
args = SFTConfig(
    output_dir="/content/drive/MyDrive/slm-post-training/sft",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=25,
    bf16=True,
    max_length=512,
    completion_only_loss=True,
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/6974 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6974 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/6974 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6974 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/6974 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/499 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/499 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/499 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/499 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/499 [00:00<?, ? examples/s]

In [27]:
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,0.571898,0.527046,0.528532,0.850117,251169.000000
200,0.563271,0.520436,0.528646,0.850265,497514.000000
300,0.537038,0.514403,0.520024,0.853829,743878.000000
400,0.520715,0.506092,0.517918,0.855014,988572.000000
500,0.437965,0.517066,0.425272,0.854687,1239156.000000
600,0.425719,0.511996,0.423220,0.855734,1486367.000000
700,0.433639,0.510746,0.429500,0.856176,1730789.000000
800,0.436334,0.510027,0.423053,0.856317,1978519.000000
872,0.389934,0.510094,0.423282,0.856569,2157824.000000


TrainOutput(global_step=872, training_loss=0.4909156469987073, metrics={'train_runtime': 776.0437, 'train_samples_per_second': 17.973, 'train_steps_per_second': 1.124, 'total_flos': 8387271846696960.0, 'train_loss': 0.4909156469987073, 'epoch': 2.0})

In [ ]:
logs = pd.DataFrame(trainer.state.log_history)

print(logs[["step", "eval_loss"]].dropna())               # every 100 steps
print(logs[["step", "loss"]].dropna().iloc[::10])         # train loss, sampled
print("best checkpoint:", trainer.state.best_model_checkpoint)

    step  eval_loss
10   100   0.527046
21   200   0.520436
32   300   0.514403
43   400   0.506092
54   500   0.517066
65   600   0.511996
76   700   0.510746
87   800   0.510027
95   872   0.510094
    step      loss
0     10  0.792729
11   110  0.543732
22   210  0.560656
33   310  0.544496
44   410  0.544343
55   510  0.410491
66   610  0.408080
77   710  0.409265
88   810  0.402653
best checkpoint: /content/drive/MyDrive/slm-post-training/sft/checkpoint-400


Eval loss is lowest at step 400 (0.506), near the end of epoch 1 (step 436). In epoch 2 the train loss drops from about 0.54 to about 0.41 while the eval loss gets no better (0.510-0.517): the model memorizes the training solutions instead of generalizing. `load_best_model_at_end` keeps checkpoint 400, which is used for everything below. A 1-epoch rerun decays the learning rate over 436 steps instead of 872, so it won't reproduce checkpoint 400 exactly.

### Saving

Both go to Google Drive: the LoRA adapter alone (~35 MB), and a merged standalone model (~1 GB) that `evaluate` loads like any Hugging Face model and that Stage 3 starts from.

In [ ]:
base = "/content/drive/MyDrive/slm-post-training"
print(trainer.state.best_model_checkpoint)

trainer.save_model(f"{base}/adapters/sft-r16-step400")

merged = trainer.model.merge_and_unload()
merged.save_pretrained(f"{base}/models/sft-merged")
tokenizer.save_pretrained(f"{base}/models/sft-merged")

/content/drive/MyDrive/slm-post-training/sft/checkpoint-400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/slm-post-training/models/sft-merged/tokenizer_config.json',
 '/content/drive/MyDrive/slm-post-training/models/sft-merged/chat_template.jinja',
 '/content/drive/MyDrive/slm-post-training/models/sft-merged/tokenizer.json')

## SFT Model Eval

The validation set is reloaded, because the training copy was turned into prompt-completion pairs. The merged model is scored zero-shot (`Question: … Answer:`) with the Stage 1 harness and settings.

In [37]:
_, val_dataset, _ = load_gsm8k()

In [38]:
questions = val_dataset["question"]
golds = val_dataset["gold"]

SFT_MODEL = f"{base}/models/sft-merged"
sft_rows = evaluate(SFT_MODEL, lambda q, _: build_prompt(q), questions, golds, stop=STOP)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/content/drive/MyDrive/slm-post-training/models/sft-merged: acc=46.5%  no-answer=1.2%  hit-limit=1.2%


In [42]:
save_results(sft_rows, STAGE_DIR / "sft_0shot_val500.json")

Zero-shot, the SFT model reaches **46.5%** accuracy on the 499 validation problems, with a **1.2%** no-answer rate and 1.2% hitting the token limit, down from 95.2% and 55.1% for the base model zero-shot. Accuracy matches the base model with 8 examples (46.9%) and its lenient zero-shot score (48.3%), as Stage 1 predicted: SFT fixed the format and stopping, not the math.

## General Check

The same 20 prompts and pass rules as Stage 1 (content and clean), with the zero-shot `Question: … Answer:` prompt the SFT model was trained on.

In [44]:
general_results = generate(SFT_MODEL, lambda q, _: build_prompt(q), [g["prompt"] for g in GENERAL], stop=STOP, max_new_tokens=256)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [45]:
save_results(general_results, STAGE_DIR / "sft_general.json")

In [46]:
general_results

[{'output': " Dear [Recipient's Name], I am writing to decline your invitation to a meeting scheduled for [Date] at [Location]. I believe that the meeting is not in my best interest and I would prefer to attend a different event. I am available to meet with you at [Time and Place] on [Date] to discuss [topic of interest]. Please let me know if you have any questions or concerns before making a decision. Thank you for your time and consideration. \nSubject: Declining Meeting Invitation\nDear [Recipient's Name], I am writing to decline your invitation to a meeting scheduled for [Date] at [Location]. I believe that the meeting is not in my best interest and I would prefer to attend a different event. I am available to meet with you at [Time and Place] on [Date] to discuss [topic of interest]. Please let me know if you have any questions or concerns before making a decision. Thank you for your time and consideration. \nSubject: Declining Meeting Invitation\nDear [Recipient's Name], I am wr

**SFT model (LoRA, step 400), zero-shot**

| ID | Category | Content | Clean | Note |
| --- | --- | --- | --- | --- |
| w1 | writing | ✗ | ✗ | Multi-sentence email (rule: 2 sentences), repeated until the limit |
| w2 | writing | ✗ | ✗ | ~200 words (rule: under ~80) |
| w3 | writing | ✗ | ✗ | List of 8 tips (rule: one tip), hit the limit |
| w4 | writing | ✓ | ✓ | |
| f1 | format | ✓ | ✓ | |
| f2 | format | ✓ | ✗ | Correct 7 days, then kept counting to 31 |
| f3 | format | ✗ | ✗ | `#### yes` |
| f4 | format | ✓ | ✗ | Appended "You are an AI assistant…" |
| t1 | translation | ✓ | ✗ | Appended "You are an AI assistant…" |
| t2 | translation | ✗ | ✗ | `####` line after the translation |
| k1 | facts | ✗ | ✗ | `#### Tokyo` |
| k2 | facts | ✗ | ✗ | `#### Jupiter` |
| k3 | facts | ✓ | ✓ | |
| k4 | facts | ✗ | ✗ | `#### Au` |
| r1 | rewrite | ✗ | ✗ | Copied a sentence, then repeated it after `####` until the limit |
| r2 | rewrite | ✗ | ✗ | Copied the input |
| r3 | rewrite | ✓ | ✗ | Appended "You are an AI assistant…" |
| j1 | judgment | ✗ | ✗ | `#### negative` |
| j2 | judgment | ✗ | ✗ | Wrong (apple) |
| j3 | judgment | ✓ | ✗ | Appended "You are an AI assistant…" |
| **Total** | | **8/20** | **3/20** | |

Against the base model (16/20 content, 6/20 clean), SFT lost half the content passes. Most of the loss is the math format leaking: 7 answers contain a `####` line (`#### Tokyo`), which fails both marks. Stopping didn't improve outside math: the model still appends "You are an AI assistant…", and on open-ended writing it repeats or lists until the token limit. It learned to stop after a `####` line, not after an answer.

## Stage 2 Summary

Validation subset, 499 problems (±2.2 pts).

| Model | Prompt | Accuracy | No-answer | General: content | General: clean |
| --- | --- | --- | --- | --- | --- |
| Base | zero-shot | 0.2% (48.3% lenient) | 95.2% | 16/20 | 6/20 |
| Base | 8-shot | 46.9% | 4.6% | - | - |
| Base + SFT | zero-shot | 46.5% | 1.2% | 8/20 | 3/20 |
| Instruct | chat, zero-shot | 53.3% (61.5% lenient) | 22.4% | 17/20 | 16/20 |

SFT changed format only. Base and SFT solve the same share of problems, about 47-48%, as Stage 1 predicted. It also cost general ability: content fell from 16/20 to 8/20, mostly from the `####` format leaking into non-math answers.

Qwen's post-training also improved the math. About 61.5% against about 48% is roughly 13 points, far outside the ±2.2 noise. This corrects the Stage 1 conclusion: Qwen's instruct model isn't just a better-formatted base model. Its post-training (much more data, math-specific tuning and RL) taught it to solve more problems.

**Target for Stage 3:** can GRPO move the model from about 47% toward about 61%? Unlike SFT, RL rewards correct answers, not imitating someone else's text, so it's the stage that could add real math ability.